In [ ]:
import logging
from pyspark.sql import SparkSession


def run_spark_job(spark):
    # TODO read format as Kafka and add various configurations
    df = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", "172.25.0.4:9092")
        .option("subscribe", "uber")
        .option("startingOffsets", "earliest")
        .option("maxOffsetsPerTrigger", 10)
        .option("maxRatePerPartition", 10)
        .option("stopGracefullyOnShutdown", "true")
        .load()
    )

    # Show schema for the incoming resources for checks
    df.printSchema()

    # agg_df = df.count()
#     agg_df = df.groupBy().count()

    # play around with processingTime to see how the progress report changes
#     query = (
#         agg_df.writeStream.trigger(processingTime="2 seconds")
#         .outputMode("complete")
#         .format("console")
#         .option("truncate", "false")
#         .start()
#     )
    
    query = df.writeStream \
        .outputMode("append") \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "172.25.0.4:9092") \
        .option("topic", "output_topic") \
        .option("checkpointLocation", "../../tmp") \
        .start()
    
    # Wait for the query to terminate
    query.awaitTermination()
    


In [ ]:
import os

JAR_FOLDER = os.path.join(os.path.abspath('../..'), 'spark/jars')

In [ ]:
spark = (
        SparkSession.builder.master("local[*]")
        .config("spark.jars", f"{JAR_FOLDER}/spark-sql-kafka-0-10_2.12-3.3.0.jar,{JAR_FOLDER}/commons-pool2-2.11.0.jar,{JAR_FOLDER}/spark-token-provider-kafka-0-10_2.12-3.3.0.jar")
        .appName("StructuredStreamingSetup")
        .getOrCreate()
    )

In [ ]:
run_spark_job(spark)